# CRISP-DM Stage 3: Modeling

1. **Kneedle Elbow Optimization**
2. **K-Means Model Training & Clustering**

In [ ]:
# Parameter Injeksi (DVC / Papermill)
k_min = 2
k_max = 10
random_state = 42
selected_features = [
    'total_koperasi',
    'rasio_nib',
    'rasio_npwp',
    'rasio_rat',
    'simpanan_pokok',
    'simpanan_wajib',
    'volume_transaksi',
    'nilai_transaksi'
]


## 1. Elbow Evaluation & Kneedle Optimization

In [ ]:
import os, pickle, pandas as pd, matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from kneed import KneeLocator
from config import CLEANED_REGENCIES_CSV, SCALED_FEATURES_CSV, MODEL_PKL, CLUSTERED_REGENCIES_CSV, FIGURES_DIR

df_reg = pd.read_csv(CLEANED_REGENCIES_CSV)
df_scaled = pd.read_csv(SCALED_FEATURES_CSV)

scaled_cols = [f"scaled_{c}" for c in selected_features if f"scaled_{c}" in df_scaled.columns]
X = df_scaled[scaled_cols].values if scaled_cols else df_scaled.values

# WCSS Calculation
k_range = list(range(k_min, k_max + 1))
wcss = [KMeans(n_clusters=k, random_state=random_state, n_init='auto').fit(X).inertia_ for k in k_range]

kn = KneeLocator(k_range, wcss, curve='convex', direction='decreasing')
optimal_k = int(kn.knee) if kn.knee is not None else 4
print(f"K Optimal Terdeteksi oleh Kneedle: K = {optimal_k}")

# Plot Kurva Elbow
os.makedirs(FIGURES_DIR, exist_ok=True)
plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', color='#1f77b4', linewidth=2, label='Inertia / WCSS')
plt.axvline(x=optimal_k, color='#d62728', linestyle='--', label=f'Optimal K = {optimal_k}')
plt.title('Evaluasi Elbow Method dengan Optimasi Kneedle')
plt.xlabel('Jumlah Klaster (K)')
plt.ylabel('WCSS')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'model_elbow_curve.png'))
plt.show()


## 2. K-Means Training & Assignment

In [ ]:
# Train Final Model
final_kmeans = KMeans(n_clusters=optimal_k, random_state=random_state, n_init=10)
df_clustered = df_reg.copy()
df_clustered['cluster_label'] = final_kmeans.fit_predict(X)

os.makedirs(os.path.dirname(MODEL_PKL), exist_ok=True)
with open(MODEL_PKL, 'wb') as f:
    pickle.dump(final_kmeans, f)

os.makedirs(os.path.dirname(CLUSTERED_REGENCIES_CSV), exist_ok=True)
df_clustered.to_csv(CLUSTERED_REGENCIES_CSV, index=False)
print("Model KMeans tersimpan dan dataset terklasterisasi berhasil dibuat.")
